# JamesExperiment: random patches from every selected slide

Filter HEST metadata and choose slides using the reference notebook's selection flow.
Every chosen slide contributes a random percentage of its patches (at least one patch per slide).
Image HDF5 files are accessed by HTTP byte ranges: select indices before reading images.
Only sampled patches are decoded and saved. Expression H5AD containers are downloaded one slide
at a time, read backed, and discarded after export; only sampled expression rows become targets.
The final image and shared-gene tensors contain exactly these sampled patches.

## 1. Dependencies and imports

Use the project's Python environment. Uncomment installation lines if needed.
Download/cleaning additionally needs anndata, h5py, NumPy and Pillow.

In [1]:
# %pip install huggingface_hub pandas torch torchvision Pillow numpy anndata h5py fsspec aiohttp
from pathlib import Path
import sys
import json
import importlib
from IPython.display import display

here = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (here, *here.parents)
     if (p / 'src' / 'hest_content_pipeline.py').is_file()
     and (p / 'JamesExperiment' / 'slide_sampling.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter inside this Hist2ST project.')
for path in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pandas as pd
import hest_content_pipeline as content
from JamesExperiment import slide_sampling, patch_sampling, compact_pipeline
importlib.reload(content)
importlib.reload(slide_sampling)
importlib.reload(patch_sampling)
importlib.reload(compact_pipeline)

METADATA_VERSION = 'HEST_v1_3_0.csv'
DATA_DIR = PROJECT_ROOT / 'data' / 'clean_data'  # Same default as reference notebook.
RUN_DIR = PROJECT_ROOT / 'outputs' / 'JamesExperiment' / 'selection'
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_DIR)
print('Selection and reports:', RUN_DIR)

Data: C:\Users\22708\Documents\BME_DataDesign\Hist2ST\Hist2ST\data\clean_data
Selection and reports: C:\Users\22708\Documents\BME_DataDesign\Hist2ST\Hist2ST\outputs\JamesExperiment\selection


In [2]:
# Run only if Hugging Face requests authentication. Never paste tokens into saved cells.
# from huggingface_hub import login
# login()

## 2. Select the data category

As in the reference: list valid categories, then enter organ, species and technology.
Comma-separated values are accepted; use "all" for no filter.
Use one species and consistent gene identifiers for meaningful shared-gene targets.
The filtered metadata snapshot is saved for later cells.

In [3]:
from huggingface_hub import hf_hub_download
from hest_download_pipeline import filter_metadata

# Fetch only the small metadata CSV once; no image/expression downloads here.
metadata_path = hf_hub_download(
    'MahmoodLab/hest', METADATA_VERSION, repo_type='dataset',
)
metadata = pd.read_csv(metadata_path, dtype={'id': str})
for column in ('organ', 'species', 'st_technology'):
    print(f"{column}: {', '.join(sorted(metadata[column].dropna().astype(str).unique()))}")

def choices(raw):
    return None if raw.strip().lower() in ('', 'all', '*') else [
        value.strip() for value in raw.split(',') if value.strip()
    ]

ORGAN = 'all'
SPECIES = 'Homo sapiens'
ST_TECHNOLOGY = 'Visium'
filtered_metadata = filter_metadata(
    metadata, organs=choices(ORGAN), species=choices(SPECIES),
    st_technologies=choices(ST_TECHNOLOGY),
)
filtered_metadata.to_csv(RUN_DIR / 'hest_reference_metadata.csv', index=False)
display(filtered_metadata.head())
print('Matching slides:', len(filtered_metadata))

C:\Users\22708\Documents\BME_DataDesign\Hist2ST\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


organ: Bladder, Bone, Bowel, Brain, Breast, Cervix, Embryo, Eye, Heart, Kidney, Kidney/Brain, Liver, Lung, Lung/Brain, Lymph node, Lymphoid, Muscle, Other, Ovary, Pancreas, Placenta, Prostate, Skin, Spinal cord, Uterus, Whole organism
species: Homo sapiens, Homo sapiens/Mus musculus, Mus musculus
st_technology: Spatial Transcriptomics, Visium, Visium HD, Visium HD 3', Xenium, Xenium 5k


,dataset_title,id,image_filename,organ,disease_state,oncotree_code,species,patient,st_technology,data_publication_date,...,treatment_comment,pixel_size_um_embedded,pixel_size_um_estimated,magnification,fullres_px_width,fullres_px_height,tissue,disease_comment,subseries,hest_version_added
0,Characterization of immune cell populations in...,TENX152,TENX152.tif,Bowel,Cancer,COAD,Homo sapiens,Patient 2,Visium,7/11/24,...,NaN,0.264583,0.273724,40x,76190,49099,Colon,NaN,"Visium CytAssist v2, Sample P2 CRC",v1_1_0
1,Spatially resolved multiomics of human cardiac...,MISC142,MISC142.tif,Heart,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,31.750063,0.454959,40x,18063,18100,right cardiac atrium,NaN,HCAHeartST10238251,v1_1_0
2,Spatially resolved multiomics of human cardiac...,MISC141,MISC141.tif,Heart,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,31.750063,0.455062,40x,17890,18098,left cardiac atrium,NaN,HCAHeartST10238252,v1_1_0
3,Spatially resolved multiomics of human cardiac...,MISC140,MISC140.tif,Heart,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,31.750063,0.455235,40x,17802,18044,apical region of left ventricle,NaN,HCAHeartST10238253,v1_1_0
4,Spatially resolved multiomics of human cardiac...,MISC139,MISC139.tif,Heart,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,31.750063,0.455581,40x,17634,17674,heart right ventricle,NaN,HCAHeartST10550730,v1_1_0


Matching slides: 421


## 3. Choose slides

Choose explicit IDs, a count/range, or all filtered slides. Every chosen slide will contribute patches.
This selection defines the slide cohort; section 4 sets the percentage of patches within each slide.

In [4]:
filtered_metadata = pd.read_csv(RUN_DIR / 'hest_reference_metadata.csv', dtype={'id': str})
available_ids = filtered_metadata['id'].tolist()
mode = '3'  # All slides matching all tissues / Homo sapiens / Visium.
if mode == '1':
    raw = input('Comma-separated slide IDs: ')
    selected_ids = list(dict.fromkeys(x.strip() for x in raw.split(',') if x.strip()))
    if not selected_ids:
        raise ValueError('Enter at least one slide ID.')
    unknown = sorted(set(selected_ids) - set(available_ids))
    if unknown:
        raise ValueError(f'IDs outside the filtered cohort: {unknown}')
elif mode == '2':
    count = int(input(f'Number of slides [available {len(available_ids)}]: '))
    start = int(input('Start at filtered slide number (1-based) [1]: ') or '1')
    if count < 1 or not 1 <= start <= len(available_ids):
        raise ValueError('Enter a positive count and valid start position.')
    if start - 1 + count > len(available_ids):
        raise ValueError('Requested range exceeds the filtered cohort.')
    selected_ids = available_ids[start - 1:start - 1 + count]
elif mode == '3':
    selected_ids = available_ids
else:
    raise ValueError('Choose 1, 2 or 3.')

selected_metadata = filtered_metadata.set_index('id').loc[selected_ids].reset_index()
selected_metadata.to_csv(RUN_DIR / 'candidate_metadata.csv', index=False)
# Download selection is written only after random sampling in section 4.
display(selected_metadata.head())
print('Candidate slides:', len(selected_metadata))

,id,dataset_title,image_filename,organ,disease_state,oncotree_code,species,patient,st_technology,data_publication_date,...,treatment_comment,pixel_size_um_embedded,pixel_size_um_estimated,magnification,fullres_px_width,fullres_px_height,tissue,disease_comment,subseries,hest_version_added
0,TENX152,Characterization of immune cell populations in...,TENX152.tif,Bowel,Cancer,COAD,Homo sapiens,Patient 2,Visium,7/11/24,...,NaN,0.264583,0.273724,40x,76190,49099,Colon,NaN,"Visium CytAssist v2, Sample P2 CRC",v1_1_0
1,MISC142,Spatially resolved multiomics of human cardiac...,MISC142.tif,Heart,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,31.750063,0.454959,40x,18063,18100,right cardiac atrium,NaN,HCAHeartST10238251,v1_1_0
2,MISC141,Spatially resolved multiomics of human cardiac...,MISC141.tif,Heart,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,31.750063,0.455062,40x,17890,18098,left cardiac atrium,NaN,HCAHeartST10238252,v1_1_0
3,MISC140,Spatially resolved multiomics of human cardiac...,MISC140.tif,Heart,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,31.750063,0.455235,40x,17802,18044,apical region of left ventricle,NaN,HCAHeartST10238253,v1_1_0
4,MISC139,Spatially resolved multiomics of human cardiac...,MISC139.tif,Heart,Healthy,NaN,Homo sapiens,,Visium,NaN,...,NaN,31.750063,0.455581,40x,17634,17674,heart right ventricle,NaN,HCAHeartST10550730,v1_1_0


Candidate slides: 421


## 4. Set patch sampling percentage BEFORE image reads

Sampling is independent within every selected slide: max(1, floor(patch_count * percentage / 100)).
The patch count comes from the remote HDF5 header, without reading the image array.
Stable slide-specific random seeds make each slide reproducible regardless of cohort order.
No slides are randomly excluded.

In [5]:
PATCH_PERCENTAGE = 7.0
SEED = 42
EXPRESSION_MODE = 'raw'  # Or 'relative_log_10000'.
if not 0 < PATCH_PERCENTAGE <= 100:
    raise ValueError('PATCH_PERCENTAGE must be greater than zero and at most 100.')
# Separate sampled-patch data from existing complete slide exports.
SAMPLED_DATA_DIR = PROJECT_ROOT / 'data' / 'JamesExperiment' / f'compact_patches_{PATCH_PERCENTAGE:g}_seed_{SEED}'
EXPORT_DIR = PROJECT_ROOT / 'outputs' / 'JamesExperiment' / f'compact_patches_{PATCH_PERCENTAGE:g}_seed_{SEED}'

selected_metadata = content.read_selected_metadata(RUN_DIR / 'candidate_metadata.csv')
slide_ids = selected_metadata['id'].tolist()
selected_metadata.to_csv(RUN_DIR / 'selected_metadata.csv', index=False)
settings = dict(
    percentage=PATCH_PERCENTAGE, seed=SEED, expression_mode=EXPRESSION_MODE,
    slide_ids=slide_ids, output_dir=str(EXPORT_DIR), data_dir=str(SAMPLED_DATA_DIR),
    sampling_unit='patch_within_slide',
    sampling_stage='before_image_reads',
    rounding='max(1, floor(N * percentage / 100))',
)
(RUN_DIR / 'sampling_settings.json').write_text(
    json.dumps(settings, indent=2), encoding='utf-8',
)
print(f'Every one of {len(slide_ids)} slides contributes {PATCH_PERCENTAGE:g}% of patches (minimum 1).')
print('Sampled data:', SAMPLED_DATA_DIR)

Every one of 421 slides contributes 7% of patches (minimum 1).
Sampled data: C:\Users\22708\Documents\BME_DataDesign\Hist2ST\Hist2ST\data\JamesExperiment\compact_patches_7_seed_42


## 5. Extract only random patches from EVERY selected slide

Remote image files must support HTTP byte ranges. There is no fallback to full image downloads.
HDF5 metadata, chunk layout and cache blocks may require neighboring bytes; transfer size is
not guaranteed to equal the patch percentage. Expression H5AD files are downloaded in full
one slide at a time because the existing AnnData backed reader requires a local file.

No whole-slide TIFFs are downloaded. No full image array is materialized or exported.
Completed slides with matching seed, percentage and dataset revision are reused.
Interrupted slide folders are preserved; use a fresh sampled-data directory when retrying them.

In [6]:
settings = patch_sampling.load_patch_settings(RUN_DIR / 'sampling_settings.json')
SAMPLED_DATA_DIR = compact_pipeline.prepare_samples(
    settings['slide_ids'], settings['data_dir'],
    percentage=settings['percentage'], seed=settings['seed'], workers=8,
)
print('Compact sampled patches ready:', SAMPLED_DATA_DIR)

Progress: 1/421 slides; 0 failed


Progress: 2/421 slides; 0 failed


Progress: 3/421 slides; 0 failed


Progress: 4/421 slides; 0 failed


Progress: 5/421 slides; 0 failed


Progress: 6/421 slides; 0 failed


... (412 outputs left out)

Progress: 419/421 slides; 0 failed


Progress: 420/421 slides; 0 failed


Progress: 421/421 slides; 0 failed


Compact sampled patches ready: C:\Users\22708\Documents\BME_DataDesign\Hist2ST\Hist2ST\data\JamesExperiment\compact_patches_7_seed_42


## 6. Build tensors from sampled patches

Use every slide and every already-sampled patch; no second sampling is performed.
The shared gene axis is intersected across these sampled patches only.
No full-cohort audit, HVG ranking, embeddings or model loading is performed.
The image and target tensors are held in CPU RAM; reduce the patch percentage if needed.
Use a fresh empty tensor output directory after an interrupted export.

In [7]:
# Uses the saved patch percentage; never samples slides a second time.
EXPORT_DIR = compact_pipeline.export_samples(RUN_DIR / 'sampling_settings.json')
print('Completed export:', EXPORT_DIR)

Exporting 66787 patches, 32 shared genes; 37.46 GiB


Completed export: C:\Users\22708\Documents\BME_DataDesign\Hist2ST\Hist2ST\outputs\JamesExperiment\compact_patches_7_seed_42


## 7. Verify saved tensors with bounded-memory batch access

The image file is a standard PyTorch tensor. Use open_batch_tensor on this Windows machine:
it opens a read-only disk view and copies only requested batches, avoiding whole-file
copy-on-write commit reservation. images[:8] returns an ordinary float32 torch tensor.
Gene targets are small enough to load normally. Rows match patch_manifest.csv;
columns match genes.json. Gene symbols, Ensembl IDs and human-reference prefixes
are aligned using the saved official HGNC annotation snapshot.

In [8]:
import torch
from JamesExperiment.tensor_stream import open_batch_tensor

settings = patch_sampling.load_patch_settings(RUN_DIR / 'sampling_settings.json')
EXPORT_DIR = Path(settings['output_dir'])
experiment = json.loads((EXPORT_DIR / 'experiment.json').read_text(encoding='utf-8'))
if not experiment.get('complete'):
    raise RuntimeError('Incomplete export; rerun into a fresh directory.')
# Release in-memory export arrays before loading if modifying the helper interactively.
images = open_batch_tensor(EXPORT_DIR / 'images.pt')
targets = torch.load(EXPORT_DIR / 'targets.pt', map_location='cpu', weights_only=True)
genes = json.loads((EXPORT_DIR / 'genes.json').read_text(encoding='utf-8'))
manifest = pd.read_csv(EXPORT_DIR / 'patch_manifest.csv')
assert list(images.shape) == experiment['images_shape']
assert list(targets.shape) == experiment['targets_shape']
assert images.shape[0] == targets.shape[0] == len(manifest)
assert targets.shape[1] == len(genes)
assert images.shape[1:] == (3, 224, 224)
assert manifest['order'].tolist() == list(range(len(manifest)))
# Avoid a redundant full-tensor scan.
print('PathFM inputs:', images.shape, images.dtype)
print('Gene targets:', targets.shape, targets.dtype)
print('First genes:', genes[:10])
display(manifest.head())
display(pd.DataFrame(targets[:5, :10].numpy(), columns=genes[:10]))

PathFM inputs: torch.Size([66787, 3, 224, 224]) torch.float32
Gene targets: torch.Size([66787, 32]) torch.float32
First genes: ['ABL1', 'AKT3', 'BAX', 'BCL2', 'BID', 'CASP3', 'CD40', 'CDC6', 'CDKN1A', 'CREBBP']


,order,slide_id,patch_name,patch_index,barcode
0,0,TENX152,TENX152_patch_00003,3,AACACGCAGATAACAA-1
1,1,TENX152,TENX152_patch_00045,45,AACGCAGGAGCAGACG-1
2,2,TENX152,TENX152_patch_00051,51,AACGGCCGTACTTCCT-1
3,3,TENX152,TENX152_patch_00110,110,AAGCCAGCTCGCGGAT-1
4,4,TENX152,TENX152_patch_00113,113,AAGCGAAGTTAGCTCA-1


,ABL1,AKT3,BAX,BCL2,BID,CASP3,CD40,CDC6,CDKN1A,CREBBP
0,3.0,0.0,1.0,2.0,1.0,1.0,1.0,0.0,2.0,1.0
1,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
2,2.0,4.0,2.0,0.0,0.0,1.0,1.0,0.0,3.0,0.0
3,27.0,1.0,36.0,0.0,15.0,7.0,1.0,13.0,22.0,25.0
4,21.0,1.0,32.0,0.0,16.0,3.0,3.0,6.0,19.0,23.0


## 8. Optional: pass a batch to GenBio-PathFM

Export is complete without this step. Uncomment to download/load the model and encode a small batch.
Install transformers==4.57.1 for the version used in the reference.

In [9]:
# %pip install transformers==4.57.1
# from transformers import AutoModel
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = AutoModel.from_pretrained('genbio-ai/genbio-pathfm', trust_remote_code=True)
# model = model.to(device).eval()
# with torch.inference_mode():
#     features = model(images[:2].to(device))
# print(features.shape)

## Verified completed run

All tissues, **Homo sapiens**, **Visium**; 7% of patches independently sampled
from every one of **421 slides**, seed 42. No slides failed.

- `images.pt`: float32 **[66787, 3, 224, 224]**, 37.45 GiB, PathFM normalized.
- `targets.pt`: float32 **[66787, 32]**, raw gene expression aligned row-for-row.
- `genes.json`: the 32 canonical genes shared by every selected source panel.

Files are in `outputs/JamesExperiment/compact_patches_7_seed_42/`.
`verification.json` records the passed coverage, reproducible patch-index,
shape, and representative image/target pairing checks. Nine regression tests passed.

Ensembl IDs, human-reference prefixes and unambiguous symbols/aliases were
aligned using the saved [official HGNC annotation](https://www.genenames.org/download/).
Some input slides already contain only about 1,000 genes, limiting the common axis.

Use the Windows-safe `open_batch_tensor` in section 7 to load image batches;
the standard whole-file private PyTorch memory map exceeds this machine's commit limit.
